In [ ]:
#hide
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
#hide
from fastbook import *
from IPython.display import display,HTML

# Approfondimento sulla PNL: RNN

In <<chapter_intro>> abbiamo visto che il deep learning può essere utilizzato per ottenere ottimi risultati con set di dati in linguaggio naturale. Il nostro esempio si basava sull'utilizzo di un modello linguistico preaddestrato e sul suo perfezionamento per classificare le recensioni. Questo esempio ha evidenziato una differenza tra l’apprendimento trasferito nella PNL e la visione artificiale: in generale nella PNL il modello preaddestrato viene addestrato su un compito diverso.

Quello che chiamiamo modello linguistico è un modello che è stato addestrato a indovinare quale sarà la parola successiva in un testo (dopo aver letto quelle precedenti). Questo tipo di compito si chiama *apprendimento auto-supervisionato*: non abbiamo bisogno di dare etichette al nostro modello, basta semplicemente alimentarlo con tanti, tanti testi. Ha un processo per ottenere automaticamente etichette dai dati, e questo compito non è banale: per indovinare correttamente la parola successiva in una frase, il modello dovrà sviluppare una comprensione della lingua inglese (o altra). L’apprendimento autosupervisionato può essere utilizzato anche in altri ambiti; ad esempio, vedere ["Apprendimento autosupervisionato e visione artificiale"](https://www.fast.ai/2020/01/13/self_supervised/) per un'introduzione alle applicazioni di visione. L'apprendimento autosupervisionato non viene solitamente utilizzato per il modello che viene addestrato direttamente, ma viene invece utilizzato per il pre-addestramento di un modello utilizzato per il trasferimento dell'apprendimento.

> gergo: Apprendimento auto-supervisionato: addestrare un modello utilizzando etichette incorporate nella variabile indipendente, anziché richiedere etichette esterne. Ad esempio, addestrando un modello a prevedere la parola successiva in un testo.

Il modello linguistico che abbiamo utilizzato in <<chapter_intro>> per classificare le recensioni di IMDb è stato pre-addestrato su Wikipedia. Abbiamo ottenuto ottimi risultati ottimizzando direttamente questo modello linguistico per un classificatore di recensioni di film, ma con un passaggio in più possiamo fare ancora meglio. L'inglese di Wikipedia è leggermente diverso dall'inglese di IMDb, quindi invece di passare direttamente al classificatore, potremmo ottimizzare il nostro modello linguistico preaddestrato al corpus di IMDb e quindi utilizzare *quello* come base per il nostro classificatore.

Anche se il nostro modello linguistico conosce le basi della lingua che stiamo utilizzando nell'attività (ad esempio, il nostro modello preaddestrato è in inglese), è utile abituarsi allo stile del corpus a cui ci rivolgiamo. Potrebbe trattarsi di un linguaggio più informale o più tecnico, con nuove parole da imparare o diversi modi di comporre le frasi. Nel caso del set di dati di IMDb, ci saranno molti nomi di registi e attori e spesso uno stile di linguaggio meno formale di quello visto su Wikipedia.

Abbiamo già visto che con fastai possiamo scaricare un modello di lingua inglese preaddestrato e utilizzarlo per ottenere risultati all'avanguardia per la classificazione NLP. (Ci aspettiamo che modelli preaddestrati in molte altre lingue siano presto disponibili: potrebbero essere disponibili nel momento in cui leggerai questo libro, in effetti.) Quindi, perché stiamo imparando come addestrare un modello linguistico in dettaglio?

Uno dei motivi, ovviamente, è che è utile comprendere le basi dei modelli che si stanno utilizzando. Ma c'è un'altra ragione molto pratica, ovvero che si ottengono risultati ancora migliori se si mette a punto il modello linguistico (basato sulla sequenza) prima di mettere a punto il modello di classificazione. Ad esempio, per l'attività di analisi del sentiment di IMDb, il set di dati include 50.000 recensioni di film aggiuntive a cui non sono associate etichette positive o negative. Dato che ci sono 25.000 recensioni etichettate nel set di training e 25.000 nel set di validazione, si ottengono complessivamente 100.000 recensioni di film. Possiamo utilizzare tutte queste revisioni per mettere a punto il modello linguistico preaddestrato, che è stato addestrato solo sugli articoli di Wikipedia; ciò si tradurrà in un modello linguistico particolarmente efficace nel prevedere la parola successiva di una recensione cinematografica.

Questo è noto come approccio ULMFit (Universal Language Model Fine-tuning). Il [articolo](https://arxiv.org/abs/1801.06146) ha mostrato che questa fase aggiuntiva di messa a punto del modello linguistico, prima di trasferire l'apprendimento in un compito di classificazione, ha portato a previsioni significativamente migliori. Utilizzando questo approccio, abbiamo tre fasi per il trasferimento dell'apprendimento nella PNL, come riassunto in <<ulmfit_process>>.

<img alt="Diagramma del processo ULMFiT" width="700" caption="Il processo ULMFiT" id="ulmfit_process" src="images/att_00027.png">

Esploreremo ora come applicare una rete neurale a questo problema di modellazione del linguaggio, utilizzando i concetti introdotti negli ultimi due capitoli. Ma prima di leggere oltre, fermati e pensa a come *tu* affronteresti questa cosa.

## Preelaborazione del testo

Non è affatto ovvio come utilizzeremo ciò che abbiamo imparato finora per costruire un modello linguistico. Le frasi possono avere lunghezze diverse e i documenti possono essere molto lunghi. Quindi, come possiamo prevedere la parola successiva di una frase utilizzando una rete neurale? Scopriamolo!

Abbiamo già visto come le variabili categoriali possano essere utilizzate come variabili indipendenti per una rete neurale. L’approccio che abbiamo adottato per una singola variabile categoriale è stato quello di:

1. Fai un elenco di tutti i possibili livelli di quella variabile categoriale (chiameremo questo elenco *vocabolario*).
1. Sostituisci ogni livello con il suo indice nel vocabolario.
1. Crea una matrice di inclusione contenente una riga per ogni livello (cioè per ogni elemento del vocabolario).
1. Utilizzare questa matrice di incorporamento come primo strato di una rete neurale. (Una matrice di incorporamento dedicata può prendere come input gli indici di vocabolario grezzi creati nel passaggio 2; questo è equivalente ma più veloce ed efficiente di una matrice che prende come input vettori con codifica a caldo che rappresentano gli indici.)

Possiamo fare quasi la stessa cosa con il testo! La novità è l'idea di sequenza. Per prima cosa concateniamo tutti i documenti nel nostro set di dati in un'unica lunga stringa e la dividiamo in parole, ottenendo un elenco molto lungo di parole (o "token"). La nostra variabile indipendente sarà la sequenza di parole che inizia con la prima parola del nostro lunghissimo elenco e termina con la penultima, e la nostra variabile dipendente sarà la sequenza di parole che inizia con la seconda parola e termina con l'ultima parola. 

Il nostro vocabolario consisterà in un mix di parole comuni che sono già nel vocabolario del nostro modello preaddestrato e nuove parole specifiche per il nostro corpus (termini cinematografici o nomi di attori, per esempio). La nostra matrice di incorporamento sarà costruita di conseguenza: per le parole che sono nel vocabolario del nostro modello preaddestrato, prenderemo la riga corrispondente nella matrice di incorporamento del modello preaddestrato; ma per le parole nuove non avremo nulla, quindi inizializzeremo semplicemente la riga corrispondente con un vettore casuale.

A ciascuno dei passaggi necessari per creare un modello linguistico è associato un gergo proveniente dal mondo dell'elaborazione del linguaggio naturale e sono disponibili classi Fastai e PyTorch per aiutare. I passaggi sono:

- Tokenizzazione:: Converti il testo in un elenco di parole (o caratteri o sottostringhe, a seconda della granularità del modello)
- Numerizzazione: crea un elenco di tutte le parole uniche che appaiono (il vocabolario) e converti ogni parola in un numero, cercando il suo indice nel vocabolario
- Creazione del caricatore dati del modello linguistico: fastai fornisce una classe `LMDataLoader` che gestisce automaticamente la creazione di una variabile dipendente che è compensata dalla variabile indipendente di un token. Gestisce anche alcuni dettagli importanti, ad esempio come mescolare i dati di addestramento in modo tale che le variabili dipendenti e indipendenti mantengano la loro struttura come richiesto
- Creazione del modello linguistico: abbiamo bisogno di un tipo speciale di modello che faccia qualcosa che non abbiamo mai visto prima: gestisca elenchi di input che potrebbero essere arbitrariamente grandi o piccoli. Esistono diversi modi per farlo; in questo capitolo utilizzeremo una *rete neurale ricorrente* (RNN). Entreremo nei dettagli di questi RNN nel <<chapter_nlp_dive>>, ma per ora puoi considerarli semplicemente come un'altra rete neurale profonda.

Diamo un'occhiata a come funziona ogni passaggio in dettaglio.

### Tokenizzazione

Quando abbiamo detto "converti il ​​testo in un elenco di parole", abbiamo tralasciato molti dettagli. Ad esempio, cosa facciamo con la punteggiatura? Come gestiamo una parola come "non fare"? È una parola o due? Che dire delle parole mediche o chimiche lunghe? Dovrebbero essere suddivisi nelle loro parti separate di significato? Che ne dici delle parole con trattino? Che ne dici di lingue come il tedesco e il polacco in cui possiamo creare parole davvero lunghe da moltissimi pezzi? Che dire delle lingue come il giapponese e il cinese che non utilizzano affatto le basi e non hanno un'idea ben definita di *parola*?

Poiché non esiste una risposta corretta a queste domande, non esiste un approccio unico alla tokenizzazione. Esistono tre approcci principali:

- Basato su parole: divide una frase in spazi, nonché applica regole specifiche della lingua per cercare di separare parti di significato anche quando non ci sono spazi (come trasformare "non" in "non"). In genere, anche i segni di punteggiatura vengono suddivisi in token separati.
- Basato su sottoparole: divide le parole in parti più piccole, in base alle sottostringhe più comuni. Ad esempio, "occasione" potrebbe essere tokenizzata come "o c ca sion".
- Basato sui caratteri: divide una frase nei suoi singoli caratteri.

Qui esamineremo la tokenizzazione di parole e sottoparole e lasceremo che sia tu a implementare la tokenizzazione basata sui caratteri nel questionario alla fine di questo capitolo.

> gergo: token: un elemento di un elenco creato dal processo di tokenizzazione. Potrebbe essere una parola, parte di una parola (una _sottoparola_) o un singolo carattere.

### Tokenizzazione delle parole con fastai

Invece di fornire i propri tokenizzatori, fastai fornisce invece un'interfaccia coerente a una gamma di tokenizzatori nelle librerie esterne. La tokenizzazione è un campo di ricerca attivo e vengono continuamente rilasciati tokenizzatori nuovi e migliorati, quindi cambiano anche le impostazioni predefinite utilizzate da fastai. Tuttavia, l'API e le opzioni non dovrebbero cambiare troppo, poiché fastai cerca di mantenere un'API coerente anche quando cambia la tecnologia sottostante.

Proviamolo con il set di dati IMDb che abbiamo utilizzato in <<chapter_intro>>:

In [ ]:
from fastai.text.all import *
path = untar_data(URLs.IMDB)

Dovremo prendere i file di testo per provare un tokenizzatore. Proprio come `get_image_files`, che abbiamo già utilizzato molte volte, ottiene tutti i file di immagine in un percorso, `get_text_files` ottiene tutti i file di testo in un percorso. Possiamo anche opzionalmente passare `folders` per restringere la ricerca a un particolare elenco di sottocartelle:

In [ ]:
files = get_text_files(path, folders = ['train', 'test', 'unsup'])

Ecco una recensione che tokenizzeremo (ne stamperemo semplicemente l'inizio qui per risparmiare spazio):

In [ ]:
txt = files[0].open().read(); txt[:75]

'This movie, which I just discovered at the video store, has apparently sit '

Mentre scriviamo questo libro, il tokenizzatore di parole inglesi predefinito per fastai utilizza una libreria chiamata *spaCy*. Ha un sofisticato motore di regole con regole speciali per URL, singole parole inglesi speciali e molto altro. Invece di usare direttamente `SpacyTokenizer`, tuttavia, useremo `WordTokenizer`, poiché punterà sempre all'attuale tokenizzatore di parole predefinito di fastai (che potrebbe non essere necessariamente spaCy, a seconda di quando stai leggendo questo).

Proviamolo. Utilizzeremo la funzione `coll_repr(collection, n)` di fastai per visualizzare i risultati. Verranno visualizzati i primi *`n`* elementi della *`collection`*, insieme alla dimensione intera: è ciò che `L` utilizza per impostazione predefinita. Tieni presente che i tokenizzatori di fastai accettano una raccolta di documenti da tokenizzare, quindi dobbiamo racchiudere `txt` in un elenco:

In [ ]:
spacy = WordTokenizer()
toks = first(spacy([txt]))
print(coll_repr(toks, 30))

(#201) ['This','movie',',','which','I','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','It',"'s",'easy','to','see'...]


Come vedi, spaCy ha principalmente separato le parole e la punteggiatura. Ma anche qui fa qualcos'altro: ha diviso "it's" in "it" e "s". Ciò ha un senso intuitivo; queste sono parole separate, davvero. La tokenizzazione è un compito sorprendentemente sottile, se si pensa a tutti i piccoli dettagli che devono essere gestiti. Fortunatamente, spaCy li gestisce abbastanza bene per noi: ad esempio, qui vediamo che "." è separato quando termina una frase, ma non con un acronimo o un numero:

In [ ]:
first(spacy(['The U.S. dollar $1 is $1.00.']))

(#9) ['The','U.S.','dollar','$','1','is','$','1.00','.']

fastai aggiunge quindi alcune funzionalità aggiuntive al processo di tokenizzazione con la classe `Tokenizer`:

In [ ]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','xxmaj','it',"'s",'easy'...]


Nota che ora ci sono alcuni token che iniziano con i caratteri "xx", che non è un prefisso di parola comune in inglese. Questi sono *gettoni speciali*.

Ad esempio, il primo elemento nell'elenco, "xxbos", è un token speciale che indica l'inizio di un nuovo testo ("BOS" è un acronimo standard della PNL che significa "inizio del flusso"). Riconoscendo questo token di inizio, il modello sarà in grado di apprendere che deve "dimenticare" ciò che è stato detto in precedenza e concentrarsi sulle parole successive.

Questi gettoni speciali non provengono direttamente da spaCy. Sono lì perché fastai li aggiunge per impostazione predefinita, applicando una serie di regole durante l'elaborazione del testo. Queste regole sono progettate per rendere più facile per un modello riconoscere le parti importanti di una frase. In un certo senso, stiamo traducendo la sequenza originale della lingua inglese in una lingua tokenizzata semplificata, una lingua progettata per essere facile da apprendere per un modello.

Ad esempio, le regole sostituiranno una sequenza di quattro punti esclamativi con uno speciale token *carattere ripetuto*, seguito dal numero quattro e quindi da un singolo punto esclamativo. In questo modo, la matrice di incorporamento del modello può codificare informazioni su concetti generali come la punteggiatura ripetuta anziché richiedere un token separato per ogni numero di ripetizioni di ciascun segno di punteggiatura. Allo stesso modo, una parola in maiuscolo verrà sostituita con uno speciale token di maiuscola, seguito dalla versione minuscola della parola. In questo modo, la matrice di incorporamento necessita solo delle versioni minuscole delle parole, risparmiando risorse di calcolo e di memoria, ma può comunque apprendere il concetto di maiuscola.

Ecco alcuni dei principali gettoni speciali che vedrai:

- `xxbos`:: Indica l'inizio di un testo (qui, una recensione)
- `xxmaj`:: Indica che la parola successiva inizia con la maiuscola (poiché abbiamo scritto tutto in minuscolo)
- `xxunk`:: Indica che la parola è sconosciuta

Per vedere le regole utilizzate, puoi controllare le regole predefinite:

In [ ]:
defaults.text_proc_rules

[<function fastai.text.core.fix_html(x)>,
 <function fastai.text.core.replace_rep(t)>,
 <function fastai.text.core.replace_wrep(t)>,
 <function fastai.text.core.spec_add_spaces(t)>,
 <function fastai.text.core.rm_useless_spaces(t)>,
 <function fastai.text.core.replace_all_caps(t)>,
 <function fastai.text.core.replace_maj(t)>,
 <function fastai.text.core.lowercase(t, add_bos=True, add_eos=False)>]

Come sempre potete consultare il codice sorgente di ciascuno di essi su un taccuino digitando:

```
??sostituisci_rep
```

Ecco un breve riepilogo di ciò che fa ciascuno:

- `fix_html`:: Sostituisce i caratteri HTML speciali con una versione leggibile (le recensioni di IMDb ne contengono parecchi)
- `replace_rep`:: Sostituisce qualsiasi carattere ripetuto tre o più volte con un token speciale per la ripetizione (`xxrep`), il numero di volte in cui viene ripetuto, quindi il carattere
- `replace_wrep`:: Sostituisce qualsiasi parola ripetuta tre o più volte con un token speciale per la ripetizione della parola (`xxwrep`), il numero di volte in cui viene ripetuta, quindi la parola
- `spec_add_spaces`:: Aggiunge spazi attorno a / e #
- `rm_useless_spaces`:: Rimuove tutte le ripetizioni del carattere spazio
- `replace_all_caps`:: riduce in minuscolo una parola scritta tutta maiuscola e aggiunge un segno speciale per tutte maiuscole ("xxup`) davanti ad essa
- `replace_maj`:: mette in minuscolo una parola in maiuscolo e aggiunge un segno speciale per la maiuscola (`xxmaj`) davanti ad essa
- `lowercase`:: riduce in minuscolo tutto il testo e aggiunge un segno speciale all'inizio (`xxbos`) e/o alla fine (`xxeos`)

Diamo un'occhiata ad alcuni di loro in azione:

In [ ]:
coll_repr(tkn('&copy;   Fast.ai www.fast.ai/INDEX'), 31)

"(#11) ['xxbos','©','xxmaj','fast.ai','xxrep','3','w','.fast.ai','/','xxup','index'...]"

Ora diamo un'occhiata a come funzionerebbe la tokenizzazione delle sottoparole.

### Tokenizzazione delle sottoparole

Oltre all'approccio di *tokenizzazione delle parole* visto nell'ultima sezione, un altro metodo di tokenizzazione popolare è la *tokenizzazione delle sottoparole*. La tokenizzazione delle parole si basa sul presupposto che gli spazi forniscano un'utile separazione dei componenti del significato in una frase. Tuttavia, questa ipotesi non è sempre appropriata. Ad esempio, considera questa frase: 我的名字是郝杰瑞 ("Il mio nome è Jeremy Howard" in cinese). Non funzionerà molto bene con un tokenizzatore di parole, perché non ci sono spazi! Lingue come il cinese e il giapponese non utilizzano gli spazi e infatti non hanno nemmeno un concetto ben definito di "parola". Esistono anche lingue, come il turco e l'ungherese, che possono aggiungere molte sottoparole insieme senza spazi, creando parole molto lunghe che includono molte informazioni separate.

Per gestire questi casi, in genere è meglio utilizzare la tokenizzazione delle sottoparole. Ciò avviene in due fasi:

1. Analizzare un corpus di documenti per trovare i gruppi di lettere più frequenti. Questi diventano il vocabolario.
2. Tokenizzare il corpus utilizzando questo vocabolario di *unità di sottoparole*.

Diamo un'occhiata a un esempio. Per il nostro corpus utilizzeremo le prime 2.000 recensioni di film:

In [ ]:
txts = L(o.open().read() for o in files[:2000])

Istanziamo il nostro tokenizer, passando la dimensione del vocabolario che vogliamo creare, e poi dobbiamo "addestrarlo". Dobbiamo cioè fargli leggere i nostri documenti e trovare le sequenze comuni di caratteri per creare il vocabolario. Questo viene fatto con "setup". Come vedremo tra poco, `setup` è uno speciale metodo fastai che viene chiamato automaticamente nelle nostre consuete pipeline di elaborazione dati. Dato che al momento stiamo facendo tutto manualmente, dobbiamo chiamarlo noi stessi. Ecco una funzione che esegue questi passaggi per una determinata dimensione del vocabolario e mostra un output di esempio:

In [ ]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

Proviamolo:

In [ ]:
subword(1000)

'▁This ▁movie , ▁which ▁I ▁just ▁dis c over ed ▁at ▁the ▁video ▁st or e , ▁has ▁a p par ent ly ▁s it ▁around ▁for ▁a ▁couple ▁of ▁years ▁without ▁a ▁dis t ri but or . ▁It'

Quando si utilizza il tokenizzatore di sottoparole di fastai, il carattere speciale ` ` rappresenta uno spazio nel testo originale.

Se utilizziamo un vocabolario più piccolo, ogni token rappresenterà meno caratteri e saranno necessari più token per rappresentare una frase:

In [ ]:
subword(200)

'▁ T h i s ▁movie , ▁w h i ch ▁I ▁ j us t ▁ d i s c o ver ed ▁a t ▁the ▁ v id e o ▁ st or e , ▁h a s'

D’altra parte, se utilizziamo un vocabolario più ampio, la maggior parte delle parole inglesi comuni finiranno nel vocabolario stesso e non ne avremo bisogno di così tante per rappresentare una frase:

In [ ]:
subword(10000)

"▁This ▁movie , ▁which ▁I ▁just ▁discover ed ▁at ▁the ▁video ▁store , ▁has ▁apparently ▁sit ▁around ▁for ▁a ▁couple ▁of ▁years ▁without ▁a ▁distributor . ▁It ' s ▁easy ▁to ▁see ▁why . ▁The ▁story ▁of ▁two ▁friends ▁living"

Scegliere la dimensione del vocabolario di una sottoparola rappresenta un compromesso: un vocabolario più grande significa meno token per frase, il che significa un addestramento più rapido, meno memoria e meno stato da ricordare per il modello; ma il lato negativo è che ciò significa matrici di inclusione più grandi, che richiedono più dati per essere apprese.

Nel complesso, la tokenizzazione delle sottoparole fornisce un modo per scalare facilmente tra la tokenizzazione dei caratteri (ovvero, utilizzando un piccolo vocabolario di sottoparole) e la tokenizzazione delle parole (ovvero, utilizzando un grande vocabolario di sottoparole) e gestisce ogni linguaggio umano senza la necessità di sviluppare algoritmi specifici della lingua. Può anche gestire altri "linguaggi" come sequenze genomiche o notazione musicale MIDI! Per questo motivo, nell’ultimo anno la sua popolarità è aumentata vertiginosamente, e sembra probabile che diventi l’approccio di tokenizzazione più comune (potrebbe già esserlo, quando leggerai questo!).

Una volta che i nostri testi sono stati suddivisi in token, dobbiamo convertirli in numeri. Lo vedremo dopo.

### Numerizzazione con fastai

La *numerazione* è il processo di mappatura dei token su numeri interi. I passaggi sono sostanzialmente identici a quelli necessari per creare una variabile "Categoria", come la variabile dipendente delle cifre in MNIST:

1. Fai un elenco di tutti i possibili livelli di quella variabile categoriale (il vocabolario).
1. Sostituisci ogni livello con il suo indice nel vocabolario.

Diamo un'occhiata a questo in azione sul testo tokenizzato in parole che abbiamo visto in precedenza:

In [ ]:
toks = tkn(txt)
print(coll_repr(tkn(txt), 31))

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at','the','video','store',',','has','apparently','sit','around','for','a','couple','of','years','without','a','distributor','.','xxmaj','it',"'s",'easy'...]


Proprio come con `SubwordTokenizer`, dobbiamo chiamare `setup` su `Numericalize`; ecco come creiamo il vocabolario. Ciò significa che avremo prima bisogno del nostro corpus tokenizzato. Poiché la tokenizzazione richiede un po' di tempo, viene eseguita in parallelo da fastai; ma per questa procedura dettagliata manuale utilizzeremo un piccolo sottoinsieme:

In [ ]:
toks200 = txts[:200].map(tkn)
toks200[0]

(#228) ['xxbos','xxmaj','this','movie',',','which','i','just','discovered','at'...]

Possiamo passarlo a `setup` per creare il nostro vocabolario:

In [ ]:
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab,20)

"(#2000) ['xxunk','xxpad','xxbos','xxeos','xxfld','xxrep','xxwrep','xxup','xxmaj','the','.',',','a','and','of','to','is','in','i','it'...]"

I nostri gettoni delle regole speciali appaiono per primi, poi ogni parola appare una volta, in ordine di frequenza. I valori predefiniti per `Numericalize` sono `min_freq=3,max_vocab=60000`. `max_vocab=60000` fa sì che fastai sostituisca tutte le parole diverse dalle più comuni 60.000 con uno speciale token *parola sconosciuta*, "xxunk". Ciò è utile per evitare di avere una matrice di incorporamento troppo grande, poiché ciò può rallentare l'addestramento e utilizzare troppa memoria e può anche significare che non ci sono dati sufficienti per addestrare rappresentazioni utili per parole rare. Tuttavia, quest'ultimo problema viene gestito meglio impostando `min_freq`; il valore predefinito "min_freq=3" significa che qualsiasi parola che appare meno di tre volte viene sostituita con "xxunk".

fastai può anche numerizzare il tuo set di dati utilizzando un vocabolario da te fornito, passando un elenco di parole come parametro "vocab".

Una volta creato il nostro oggetto `Numericalize`, possiamo usarlo come se fosse una funzione:

In [ ]:
nums = num(toks)[:20]; nums

tensor([  2,   8,  21,  28,  11,  90,  18,  59,   0,  45,   9, 351, 499,  11,  72, 533, 584, 146,  29,  12])

Questa volta, i nostri token sono stati convertiti in un tensore di numeri interi che il nostro modello può ricevere. Possiamo verificare che corrispondano al testo originale:

In [ ]:
' '.join(num.vocab[o] for o in nums)

'xxbos xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a'

Ora che abbiamo i numeri, dobbiamo inserirli in batch per il nostro modello.

### Mettere i nostri testi in batch per un modello linguistico

Quando abbiamo a che fare con le immagini, dovevamo ridimensionarle tutte alla stessa altezza e larghezza prima di raggrupparle insieme in un mini-batch in modo che potessero impilarle insieme in modo efficiente in un unico tensore. Qui sarà un po' diverso, perché non è possibile semplicemente ridimensionare il testo alla lunghezza desiderata. Inoltre, vogliamo che il nostro modello linguistico legga il testo in ordine, in modo da poter prevedere in modo efficiente quale sarà la parola successiva. Ciò significa che ogni nuovo lotto dovrebbe iniziare esattamente da dove si era interrotto il precedente.

Supponiamo di avere il seguente testo:

> : In questo capitolo riprenderemo l'esempio di classificazione delle recensioni cinematografiche che abbiamo studiato nel capitolo 1 e scaveremo più a fondo sotto la superficie. Per prima cosa esamineremo le fasi di elaborazione necessarie per convertire il testo in numeri e come personalizzarlo. In questo modo, avremo un altro esempio del preprocessore utilizzato nell'API del blocco dati.\nPoi studieremo come costruire un modello linguistico e addestrarlo per un po'.

Il processo di tokenizzazione aggiungerà token speciali e gestirà la punteggiatura per restituire questo testo:

>: xxbos xxmaj In questo capitolo riprenderemo l'esempio di classificazione delle recensioni cinematografiche che abbiamo studiato nel capitolo 1 e scaveremo più in profondità sotto la superficie. xxmaj per prima cosa esamineremo le fasi di elaborazione necessarie per convertire il testo in numeri e come personalizzarlo. xxmaj così facendo, avremo un altro esempio del preprocessore utilizzato nel blocco dati xxup api. \n xxmaj poi studieremo come costruire un modello linguistico e addestrarlo per un po'.

Ora abbiamo 90 gettoni, separati da spazi. Diciamo che vogliamo una dimensione batch pari a 6. Dobbiamo suddividere questo testo in 6 parti contigue di lunghezza 15:

In [ ]:
#hide_input
stream = "In this chapter, we will go back over the example of classifying movie reviews we studied in chapter 1 and dig deeper under the surface. First we will look at the processing steps necessary to convert text into numbers and how to customize it. By doing this, we'll have another example of the PreProcessor used in the data block API.\nThen we will study how we build a language model and train it for a while."
tokens = tkn(stream)
bs,seq_len = 6,15
d_tokens = np.array([tokens[i*seq_len:(i+1)*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter,",",we,will,go,back,over,the,example,of,classifying
movie,reviews,we,studied,in,chapter,1,and,dig,deeper,under,the,surface,.,xxmaj
first,we,will,look,at,the,processing,steps,necessary,to,convert,text,into,numbers,and
how,to,customize,it,.,xxmaj,by,doing,this,",",we,'ll,have,another,example
of,the,preprocessor,used,in,the,data,block,xxup,api,.,\n,xxmaj,then,we
will,study,how,we,build,a,language,model,and,train,it,for,a,while,.


In un mondo perfetto, potremmo quindi dare questo lotto al nostro modello. Ma questo approccio non è scalabile, perché al di fuori di questo esempio giocattolo è improbabile che un singolo batch contenente tutti i testi possa entrare nella memoria della nostra GPU (qui abbiamo 90 token, ma tutte le recensioni di IMDb insieme danno diversi milioni).

Quindi, dobbiamo dividere questo array più finemente in sottoarray di lunghezza di sequenza fissa. È importante mantenere l'ordine all'interno e attraverso questi sottoarray, perché utilizzeremo un modello che mantiene uno stato in modo da ricordare ciò che ha letto in precedenza quando si prevede ciò che verrà dopo. 

Tornando al nostro esempio precedente con 6 batch di lunghezza 15, se scegliessimo una lunghezza di sequenza pari a 5, ciò significherebbe prima alimentare il seguente array:

In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15:i*15+seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

xxbos,xxmaj,in,this,chapter
movie,reviews,we,studied,in
first,we,will,look,at
how,to,customize,it,.
of,the,preprocessor,used,in
will,study,how,we,build


Allora questo:

In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+seq_len:i*15+2*seq_len] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

",",we,will,go,back
chapter,1,and,dig,deeper
the,processing,steps,necessary,to
xxmaj,by,doing,this,","
the,data,block,xxup,api
a,language,model,and,train


E infine:

In [ ]:
#hide_input
bs,seq_len = 6,5
d_tokens = np.array([tokens[i*15+10:i*15+15] for i in range(bs)])
df = pd.DataFrame(d_tokens)
display(HTML(df.to_html(index=False,header=None)))

over,the,example,of,classifying
under,the,surface,.,xxmaj
convert,text,into,numbers,and
we,'ll,have,another,example
.,\n,xxmaj,then,we
it,for,a,while,.


Tornando al nostro dataset delle recensioni cinematografiche, il primo passo è trasformare i singoli testi in un flusso concatenandoli insieme. Come per le immagini, è meglio randomizzare l'ordine degli input, quindi all'inizio di ogni epoca mescoleremo le voci per creare un nuovo flusso (miscoliamo l'ordine dei documenti, non quello delle parole al loro interno, altrimenti i testi non avrebbero più senso!).

Quindi dividiamo questo flusso in un certo numero di batch (che è la nostra *dimensione batch*). Ad esempio, se il flusso ha 50.000 token e impostiamo una dimensione batch pari a 10, questo ci darà 10 mini-stream di 5.000 token. L'importante è preservare l'ordine dei token (quindi da 1 a 5.000 per il primo mini-stream, poi da 5.001 a 10.000...), perché vogliamo che il modello legga righe di testo continue (come nell'esempio precedente). Un token `xxbos` viene aggiunto all'inizio di ciascuno durante la preelaborazione, in modo che il modello sappia quando legge lo stream quando inizia una nuova voce.

Quindi, per ricapitolare, in ogni momento mescoliamo la nostra raccolta di documenti e li concateniamo in un flusso di token. Quindi tagliamo il flusso in un batch di mini-flussi consecutivi di dimensione fissa. Il nostro modello leggerà quindi i mini-flussi in ordine e, grazie a uno stato interno, produrrà la stessa attivazione qualunque sia la lunghezza della sequenza scelta.

Tutto questo viene fatto dietro le quinte dalla libreria fastai quando creiamo un `LMDataLoader`. Lo facciamo applicando prima il nostro oggetto `Numericalize` ai testi tokenizzati:

In [ ]:
nums200 = toks200.map(num)

e poi passandolo a "LMDataLoader":

In [ ]:
dl = LMDataLoader(nums200)

Confermiamo che questo dia i risultati attesi, prendendo il primo lotto:

In [ ]:
x,y = first(dl)
x.shape,y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

e poi guardando la prima riga della variabile indipendente, che dovrebbe essere l'inizio del primo testo:

In [ ]:
' '.join(num.vocab[o] for o in x[0][:20])

'xxbos xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a'

La variabile dipendente è la stessa cosa compensata da un token:

In [ ]:
' '.join(num.vocab[o] for o in y[0][:20])

'xxmaj this movie , which i just xxunk at the video store , has apparently sit around for a couple'

Questo conclude tutte le fasi di preelaborazione che dobbiamo applicare ai nostri dati. Ora siamo pronti per addestrare il nostro classificatore di testo.

## Addestramento di un classificatore di testo

Come abbiamo visto all'inizio di questo capitolo, ci sono due passaggi per addestrare un classificatore di testo all'avanguardia utilizzando l'apprendimento di trasferimento: prima dobbiamo ottimizzare il nostro modello linguistico preaddestrato su Wikipedia al corpus delle recensioni di IMDb, quindi possiamo utilizzare quel modello per addestrare un classificatore.

Come al solito, iniziamo con l'assemblaggio dei nostri dati.

### Modello linguistico che utilizza DataBlock

fastai gestisce automaticamente la tokenizzazione e la numerizzazione quando "TextBlock" viene passato a "DataBlock". Tutti gli argomenti che possono essere passati a "Tokenize" e "Numericalize" possono essere passati anche a "TextBlock". Nel prossimo capitolo discuteremo i modi più semplici per eseguire ciascuno di questi passaggi separatamente, per facilitare il debug, ma puoi sempre eseguire il debug eseguendoli manualmente su un sottoinsieme dei tuoi dati, come mostrato nelle sezioni precedenti. E non dimenticare il pratico metodo "summary" di `DataBlock`, che è molto utile per il debug dei problemi relativi ai dati.

Ecco come utilizziamo `TextBlock` per creare un modello linguistico, utilizzando le impostazioni predefinite di fastai:

In [ ]:
get_imdb = partial(get_text_files, folders=['train', 'test', 'unsup'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True),
    get_items=get_imdb, splitter=RandomSplitter(0.1)
).dataloaders(path, path=path, bs=128, seq_len=80)

Una cosa che è diversa dai tipi precedenti che abbiamo usato in `DataBlock` è che non stiamo semplicemente usando la classe direttamente (ad esempio, `TextBlock(...)`, ma stiamo invece chiamando un *metodo di classe*. Un metodo di classe è un metodo Python che, come suggerisce il nome, appartiene a una *classe* piuttosto che a un *oggetto*. (Assicuratevi di cercare online ulteriori informazioni sui metodi di classe se non li conoscete, poiché sono comunemente usati in molte librerie Python e applicazioni; li abbiamo usati alcune volte in precedenza nel libro, ma non abbiamo richiamato l'attenzione su di essi.) La ragione per cui `TextBlock` è speciale è che l'impostazione del vocabolario del numerizzatore può richiedere molto tempo (dobbiamo leggere e tokenizzare ogni documento per ottenere il vocabolario). Per essere il più efficiente possibile, esegue alcune ottimizzazioni: 

- Salva i documenti tokenizzati in una cartella temporanea, quindi non è necessario tokenizzarli più di una volta
- Esegue più processi di tokenizzazione in parallelo, per sfruttare le CPU del tuo computer

Dobbiamo dire a `TextBlock` come accedere ai testi, in modo che possa eseguire questa preelaborazione iniziale: ecco cosa fa `from_folder`.

`show_batch` funziona quindi nel solito modo:

In [ ]:
dls_lm.show_batch(max_n=2)

,text,text_
0,"xxbos xxmaj it 's awesome ! xxmaj in xxmaj story xxmaj mode , your going from punk to pro . xxmaj you have to complete goals that involve skating , driving , and walking . xxmaj you create your own skater and give it a name , and you can make it look stupid or realistic . xxmaj you are with your friend xxmaj eric throughout the game until he betrays you and gets you kicked off of the skateboard","xxmaj it 's awesome ! xxmaj in xxmaj story xxmaj mode , your going from punk to pro . xxmaj you have to complete goals that involve skating , driving , and walking . xxmaj you create your own skater and give it a name , and you can make it look stupid or realistic . xxmaj you are with your friend xxmaj eric throughout the game until he betrays you and gets you kicked off of the skateboard xxunk"
1,"what xxmaj i 've read , xxmaj death xxmaj bed is based on an actual dream , xxmaj george xxmaj barry , the director , successfully transferred dream to film , only a genius could accomplish such a task . \n\n xxmaj old mansions make for good quality horror , as do portraits , not sure what to make of the killer bed with its killer yellow liquid , quite a bizarre dream , indeed . xxmaj also , this","xxmaj i 've read , xxmaj death xxmaj bed is based on an actual dream , xxmaj george xxmaj barry , the director , successfully transferred dream to film , only a genius could accomplish such a task . \n\n xxmaj old mansions make for good quality horror , as do portraits , not sure what to make of the killer bed with its killer yellow liquid , quite a bizarre dream , indeed . xxmaj also , this is"


Ora che i nostri dati sono pronti, possiamo mettere a punto il modello linguistico preaddestrato.

### Messa a punto del modello linguistico

Per convertire gli indici di parole intere in attivazioni che possiamo utilizzare per la nostra rete neurale, utilizzeremo gli incorporamenti, proprio come abbiamo fatto per il filtraggio collaborativo e la modellazione tabulare. Quindi inseriremo questi incorporamenti in una *rete neurale ricorrente* (RNN), utilizzando un'architettura chiamata *AWD-LSTM* (ti mostreremo come scrivere un modello del genere da zero nel <<chapter_nlp_dive>>). Come discusso in precedenza, gli incorporamenti nel modello preaddestrato vengono uniti con incorporamenti casuali aggiunti per parole che non erano nel vocabolario di preaddestramento. Questo viene gestito automaticamente all'interno di `lingual_model_learner`:

In [ ]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3, 
    metrics=[accuracy, Perplexity()]).to_fp16()

La funzione di perdita utilizzata per impostazione predefinita è la perdita di entropia incrociata, poiché essenzialmente abbiamo un problema di classificazione (le diverse categorie sono le parole nel nostro vocabolario). La metrica *perplessità* qui utilizzata è spesso utilizzata in PNL per i modelli linguistici: è l'esponenziale della perdita (ad esempio, `torch.exp(cross_entropy)`). Includiamo anche la metrica dell'accuratezza, per vedere quante volte il nostro modello ha ragione quando tenta di prevedere la parola successiva, poiché l'entropia incrociata (come abbiamo visto) è difficile da interpretare e ci dice di più sulla confidenza del modello che sulla sua accuratezza.

Torniamo al diagramma del processo dall'inizio di questo capitolo. La prima freccia è stata completata per noi e resa disponibile come modello preaddestrato in fastai, e abbiamo appena creato "DataLoaders" e "Learner" per la seconda fase. Ora siamo pronti per mettere a punto il nostro modello linguistico!

<img alt="Diagramma del processo ULMFiT" width="450" ​​src="images/att_00027.png">

L'addestramento di ciascuna epoca richiede un po' di tempo, quindi salveremo i risultati del modello intermedio durante il processo di addestramento. Dato che `fine_tune` non lo fa per noi, useremo `fit_one_cycle`. Proprio come `vision_learner`, `lingual_model_learner` chiama automaticamente `freeze` quando si utilizza un modello preaddestrato (che è l'impostazione predefinita), quindi addestrerà solo gli incorporamenti (l'unica parte del modello che contiene pesi inizializzati in modo casuale, ovvero incorporamenti per parole che sono nel nostro vocabolario IMDb, ma non sono nel vocabolario del modello preaddestrato):

In [ ]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,4.120048,3.912788,0.299565,50.038246,11:39


Questo modello richiede un po' di tempo per essere addestrato, quindi è una buona opportunità per parlare del salvataggio dei risultati intermedi.

### Salvataggio e caricamento dei modelli

Puoi facilmente salvare lo stato del tuo modello in questo modo:

In [ ]:
learn.save('1epoch')

Questo creerà un file in `learn.path/models/` chiamato *1epoch.pth*. Se desideri caricare il tuo modello su un'altra macchina dopo aver creato il tuo "Studente" nello stesso modo, o riprendere l'allenamento in un secondo momento, puoi caricare il contenuto di questo file con:

In [ ]:
learn = learn.load('1epoch')

Una volta completata la formazione iniziale, possiamo continuare a mettere a punto il modello dopo lo scongelamento:

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(10, 2e-3)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,3.893486,3.772820,0.317104,43.502548,12:37
1,3.820479,3.717197,0.323790,41.148880,12:30
2,3.735622,3.659760,0.330321,38.851997,12:09
3,3.677086,3.624794,0.333960,37.516987,12:12
4,3.636646,3.601300,0.337017,36.645859,12:05
5,3.553636,3.584241,0.339355,36.026001,12:04
6,3.507634,3.571892,0.341353,35.583862,12:08
7,3.444101,3.565988,0.342194,35.374371,12:08
8,3.398597,3.566283,0.342647,35.384815,12:11
9,3.375563,3.568166,0.342528,35.451500,12:05


Fatto ciò, salviamo tutto il nostro modello tranne lo strato finale che converte le attivazioni in probabilità di scegliere ciascun token nel nostro vocabolario. Il modello che non include lo strato finale è chiamato *codificatore*. Possiamo salvarlo con `save_encoder`:

In [ ]:
learn.save_encoder('finetuned')

> gergo: Codificatore: il modello che non include gli strati finali specifici dell'attività. Questo termine significa più o meno la stessa cosa di _corpo_ quando applicato alle CNN di visione, ma "codificatore" tende ad essere più utilizzato per la PNL e i modelli generativi.

Ciò completa la seconda fase del processo di classificazione del testo: la messa a punto del modello linguistico. Ora possiamo usarlo per ottimizzare un classificatore utilizzando le etichette di sentiment di IMDb.

### Generazione di testo

Prima di passare alla messa a punto del classificatore, proviamo rapidamente qualcosa di diverso: utilizzare il nostro modello per generare recensioni casuali. Poiché è addestrato a indovinare qual è la parola successiva della frase, possiamo utilizzare il modello per scrivere nuove recensioni:

In [ ]:
TEXT = "I liked this movie because"
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75) 
         for _ in range(N_SENTENCES)]

In [ ]:
print("\n".join(preds))

i liked this movie because of its story and characters . The story line was very strong , very good for a sci - fi film . The main character , Alucard , was very well developed and brought the whole story
i liked this movie because i like the idea of the premise of the movie , the ( very ) convenient virus ( which , when you have to kill a few people , the " evil " machine has to be used to protect


Come puoi vedere, aggiungiamo un po' di casualità (scegliamo una parola casuale in base alle probabilità restituite dal modello) in modo da non ottenere esattamente la stessa recensione due volte. Il nostro modello non ha alcuna conoscenza programmata della struttura di una frase o delle regole grammaticali, tuttavia ha chiaramente imparato molto sulle frasi inglesi: possiamo vedere che usa correttamente le maiuscole (*I* viene semplicemente trasformata in *i* perché le nostre regole richiedono due o più caratteri per considerare una parola come maiuscola, quindi è normale vederla minuscola) e utilizza il tempo coerente. La recensione generale ha senso a prima vista, ed è solo se leggi attentamente che puoi notare che qualcosa non va. Non male per una modella addestrata in un paio d'ore! 

Ma il nostro obiettivo finale non era addestrare un modello per generare recensioni, ma classificarle... quindi utilizziamo questo modello per fare proprio questo.

### Creazione dei dataloader del classificatore

Stiamo ora passando dalla messa a punto del modello linguistico alla messa a punto del classificatore. Ricapitolando, un modello linguistico prevede la parola successiva di un documento, quindi non necessita di etichette esterne. Un classificatore, tuttavia, prevede qualche etichetta esterna: nel caso di IMDb, è il sentimento di un documento.

Ciò significa che la struttura del nostro "DataBlock" per la classificazione PNL sembrerà molto familiare. In realtà è quasi lo stesso che abbiamo visto per i numerosi set di dati di classificazione delle immagini con cui abbiamo lavorato:

In [ ]:
dls_clas = DataBlock(
    blocks=(TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
    get_y = parent_label,
    get_items=partial(get_text_files, folders=['train', 'test']),
    splitter=GrandparentSplitter(valid_name='test')
).dataloaders(path, path=path, bs=128, seq_len=72)

Proprio come con la classificazione delle immagini, "show_batch" mostra la variabile dipendente (sentimento, in questo caso) con ciascuna variabile indipendente (testo della recensione del film):

In [ ]:
dls_clas.show_batch(max_n=3)

,text,category
0,"xxbos i rate this movie with 3 skulls , only coz the girls knew how to scream , this could 've been a better movie , if actors were better , the twins were xxup ok , i believed they were evil , but the eldest and youngest brother , they sucked really bad , it seemed like they were reading the scripts instead of acting them … . spoiler : if they 're vampire 's why do they freeze the blood ? vampires ca n't drink frozen blood , the sister in the movie says let 's drink her while she is alive … .but then when they 're moving to another house , they take on a cooler they 're frozen blood . end of spoiler \n\n it was a huge waste of time , and that made me mad coz i read all the reviews of how",neg
1,"xxbos i have read all of the xxmaj love xxmaj come xxmaj softly books . xxmaj knowing full well that movies can not use all aspects of the book , but generally they at least have the main point of the book . i was highly disappointed in this movie . xxmaj the only thing that they have in this movie that is in the book is that xxmaj missy 's father comes to xxunk in the book both parents come ) . xxmaj that is all . xxmaj the story line was so twisted and far fetch and yes , sad , from the book , that i just could n't enjoy it . xxmaj even if i did n't read the book it was too sad . i do know that xxmaj pioneer life was rough , but the whole movie was a downer . xxmaj the rating",neg
2,"xxbos xxmaj this , for lack of a better term , movie is lousy . xxmaj where do i start … … \n\n xxmaj cinemaphotography - xxmaj this was , perhaps , the worst xxmaj i 've seen this year . xxmaj it looked like the camera was being tossed from camera man to camera man . xxmaj maybe they only had one camera . xxmaj it gives you the sensation of being a volleyball . \n\n xxmaj there are a bunch of scenes , haphazardly , thrown in with no continuity at all . xxmaj when they did the ' split screen ' , it was absurd . xxmaj everything was squished flat , it looked ridiculous . \n\n xxmaj the color tones were way off . xxmaj these people need to learn how to balance a camera . xxmaj this ' movie ' is poorly made , and",neg


Osservando la definizione di "DataBlock", ogni pezzo è familiare ai precedenti blocchi di dati che abbiamo creato, con due importanti eccezioni:

- `TextBlock.from_folder` non ha più il parametro `is_lm=True`.
- Passiamo il `vocab` che abbiamo creato per la messa a punto del modello linguistico.

Il motivo per cui passiamo il `vocab` del modello linguistico è assicurarci di utilizzare la stessa corrispondenza tra token e indice. Altrimenti gli incorporamenti che abbiamo imparato nel nostro modello linguistico perfezionato non avranno alcun senso per questo modello e la fase di perfezionamento non sarà di alcuna utilità.

Passando `is_lm=False` (o non passando affatto `is_lm`, poiché il valore predefinito è "False") diciamo a `TextBlock` che abbiamo dati etichettati regolarmente, anziché utilizzare i token successivi come etichette. C'è però una sfida che dobbiamo affrontare, che ha a che fare con la raccolta di più documenti in un mini-batch. Vediamo con un esempio, provando a creare un mini-batch contenente i primi 10 documenti. Per prima cosa li numeriamo:

In [ ]:
nums_samp = toks200[:10].map(num)

Diamo ora un'occhiata a quanti token ha ciascuna di queste 10 recensioni di film:

In [ ]:
nums_samp.map(len)

(#10) [228,238,121,290,196,194,533,124,581,155]

Ricorda, PyTorch `DataLoader` deve raccogliere tutti gli elementi in un batch in un singolo tensore e un singolo tensore ha una forma fissa (ovvero, ha una lunghezza particolare su ogni asse e tutti gli elementi devono essere coerenti). Dovrebbe suonarti familiare: abbiamo avuto lo stesso problema con le immagini. In quel caso, abbiamo utilizzato il ritaglio, il riempimento e/o lo schiacciamento per rendere tutti gli input della stessa dimensione. Il ritaglio potrebbe non essere una buona idea per i documenti, perché sembra probabile che rimuoveremmo alcune informazioni chiave (detto questo, lo stesso problema vale per le immagini, e lì usiamo il ritaglio; l'aumento dei dati non è stato ancora ben esplorato per la PNL, quindi forse ci sono effettivamente opportunità di utilizzare il ritaglio anche in PNL!). Non puoi davvero "schiacciare" un documento. Quindi rimane l'imbottitura!

Espanderemo i testi più brevi per renderli tutti della stessa dimensione. Per fare ciò, utilizziamo uno speciale token di riempimento che verrà ignorato dal nostro modello. Inoltre, per evitare problemi di memoria e migliorare le prestazioni, raggrupperemo testi che hanno all'incirca la stessa lunghezza (con qualche rimescolamento per il set di addestramento). Lo facciamo (approssimativamente, per il set di addestramento) ordinando i documenti per lunghezza prima di ciascuna epoca. Il risultato di ciò è che i documenti raccolti in un unico batch tenderanno ad avere lunghezze simili. Non riempiremo ogni batch con la stessa dimensione, ma utilizzeremo invece la dimensione del documento più grande in ogni batch come dimensione target. (È possibile fare qualcosa di simile con le immagini, il che è particolarmente utile per immagini rettangolari di dimensioni irregolari, ma al momento in cui scriviamo nessuna libreria fornisce ancora un buon supporto per questo, e non ci sono documenti che lo coprano. È qualcosa che stiamo pianificando di aggiungere presto a fastai, quindi tieni d'occhio il sito web del libro; aggiungeremo informazioni a riguardo non appena funzionerà bene.)

L'ordinamento e il riempimento vengono eseguiti automaticamente dall'API del blocco dati per noi quando utilizziamo un `TextBlock`, con `is_lm=False`. (Non abbiamo lo stesso problema per i dati del modello linguistico, poiché prima concateniamo insieme tutti i documenti e poi li dividiamo in sezioni di uguali dimensioni.)

Ora possiamo creare un modello per classificare i nostri testi:

In [ ]:
learn = text_classifier_learner(dls_clas, AWD_LSTM, drop_mult=0.5, 
                                metrics=accuracy).to_fp16()

Il passaggio finale prima dell'addestramento del classificatore consiste nel caricare il codificatore dal nostro modello linguistico ottimizzato. Usiamo `load_encoder` invece di `load` perché abbiamo a disposizione solo pesi pre-addestrati per il codificatore; `load` per impostazione predefinita solleva un'eccezione se viene caricato un modello incompleto:

In [ ]:
learn = learn.load_encoder('finetuned')

### Ottimizzazione del classificatore

L'ultimo passo è allenarsi con tassi di apprendimento discriminativi e *sblocco graduale*. Nella visione artificiale spesso sblocchiamo il modello tutto in una volta, ma per i classificatori PNL scopriamo che scongelare pochi strati alla volta fa davvero la differenza:

In [ ]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.347427,0.184480,0.929320,00:33


In una sola epoca otteniamo lo stesso risultato del nostro allenamento in <<chapter_intro>>: non male! Possiamo passare `-2` a `freeze_to` per congelare tutto tranne gli ultimi due gruppi di parametri:

In [ ]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.247763,0.171683,0.934640,00:37


Poi possiamo sbloccarci ancora un po' e continuare l'allenamento:

In [ ]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.193377,0.156696,0.941200,00:45


E infine, il modello intero!

In [ ]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.172888,0.153770,0.943120,01:01
1,0.161492,0.155567,0.942640,00:57


Abbiamo raggiunto una precisione del 94,3%, una prestazione all'avanguardia solo tre anni fa. Addestrando un altro modello su tutti i testi letti al contrario e facendo la media delle previsioni di questi due modelli, possiamo arrivare anche a una precisione del 95,1%, che era lo stato dell’arte introdotto dal documento ULMFiT. È stato battuto solo pochi mesi fa, mettendo a punto un modello molto più grande e utilizzando costose tecniche di aumento dei dati (traduzione di frasi in un'altra lingua e viceversa, utilizzando un altro modello per la traduzione).

L'utilizzo di un modello preaddestrato ci ha permesso di creare un modello linguistico ottimizzato che fosse piuttosto potente, per generare recensioni false o aiutarci a classificarle. È roba entusiasmante, ma è bene ricordare che questa tecnologia può essere utilizzata anche per scopi dannosi.

## Disinformazione e modelli linguistici

Anche semplici algoritmi basati su regole, prima dell’avvento dei modelli linguistici di deep learning ampiamente disponibili, potevano essere utilizzati per creare account fraudolenti e cercare di influenzare i politici. Jeff Kao, ora giornalista computazionale presso ProPublica, ha analizzato i commenti inviati alla Federal Communications Commission (FCC) degli Stati Uniti in merito a una proposta del 2017 per abrogare la neutralità della rete. Nel suo articolo ["More than a Million Pro-Repeal Net Neutrality Comments Were Likely Faked"](https://hackernoon.com/more-than-a-million-pro-repeal-net-neutrality-comments-were-likely-faked-e9f0e3ed36a6), riferisce di come ha scoperto un ampio gruppo di commenti contrari alla neutralità della rete che sembravano essere stati generati da una sorta di posta in stile Mad Libs unire. In <<disinformazione>>, i commenti falsi sono stati utilmente codificati a colori da Kao per evidenziare la loro natura stereotipata.

<img src="images/ethics/image16.png" width="700" id="disinformation" caption="Commenti ricevuti dalla FCC durante il dibattito sulla neutralità della rete">

Kao ha stimato che "meno di 800.000 degli oltre 22 milioni di commenti... potrebbero essere considerati veramente unici" e che "più del 99% dei commenti veramente unici erano a favore del mantenimento della neutralità della rete".

Considerati i progressi nella modellazione linguistica verificatisi a partire dal 2017, tali campagne fraudolente potrebbero essere quasi impossibili da individuare ora.  Ora hai tutti gli strumenti necessari a tua disposizione per creare un modello linguistico avvincente, ovvero qualcosa che possa generare testo credibile e appropriato al contesto. Non sarà necessariamente perfettamente accurato o corretto, ma sarà plausibile. Pensa a cosa significherebbe questa tecnologia se messa insieme ai tipi di campagne di disinformazione di cui abbiamo appreso negli ultimi anni. Dai un'occhiata al dialogo Reddit mostrato in <<ethics_reddit>>, dove un modello linguistico basato sull'algoritmo GPT-2 di OpenAI sta conversando con se stesso sull'opportunità che il governo degli Stati Uniti debba tagliare le spese per la difesa.

<img src="images/ethics/image14.png" id="ethics_reddit" caption="Un algoritmo che parla con se stesso su Reddit" alt="Un algoritmo che parla con se stesso su Reddit" width="600">

In questo caso, è stato detto esplicitamente che è stato utilizzato un algoritmo, ma immagina cosa accadrebbe se un cattivo attore decidesse di rilasciare un simile algoritmo sui social network. Potrebbero farlo lentamente e con attenzione, consentendo all’algoritmo di sviluppare gradualmente follower e fiducia nel tempo. Non ci vorrebbero molte risorse per avere letteralmente milioni di account che lo fanno. In una situazione del genere potremmo facilmente immaginare di arrivare al punto in cui la stragrande maggioranza dei discorsi online proveniva dai bot, e nessuno avrebbe idea che ciò stesse accadendo.

Stiamo già iniziando a vedere esempi di machine learning utilizzati per generare identità. Ad esempio, <<katie_jones>> mostra un profilo LinkedIn per Katie Jones.

<img src="images/ethics/image15.jpeg" width="400" id="katie_jones" caption="Profilo LinkedIn di Katie Jones">

Katie Jones era collegata su LinkedIn a diversi membri dei principali think tank di Washington. Ma lei non esisteva. L'immagine che vedi è stata generata automaticamente da una rete avversaria generativa e qualcuno di nome Katie Jones, in effetti, non si è laureato al Centro per gli studi strategici e internazionali.

Molte persone presumono o sperano che gli algoritmi vengano in nostra difesa, che svilupperemo algoritmi di classificazione in grado di riconoscere automaticamente i contenuti generati automaticamente. Il problema, tuttavia, è che questa sarà sempre una corsa agli armamenti, in cui algoritmi di classificazione (o discriminatori) migliori potranno essere utilizzati per creare algoritmi di generazione migliori.

## Conclusione

In questo capitolo abbiamo esplorato l'ultima applicazione immediatamente coperta dalla libreria fastai: text. Abbiamo visto due tipi di modelli: modelli linguistici che possono generare testi e un classificatore che determina se una recensione è positiva o negativa. Per costruire un classificatore all'avanguardia, abbiamo utilizzato un modello linguistico preaddestrato, lo abbiamo adattato al corpus del nostro compito, quindi abbiamo utilizzato il suo corpo (il codificatore) con una nuova testa per eseguire la classificazione.

Prima di concludere questa sezione, daremo un'occhiata a come la libreria fastai può aiutarti a assemblare i tuoi dati per i tuoi problemi specifici.

## Questionario

1. Che cos'è l'"apprendimento autosupervisionato"?
1. Cos'è un "modello linguistico"?
1. Perché un modello linguistico è considerato auto-supervisionato?
1. A cosa servono solitamente i modelli autosupervisionati?
1. Perché perfezioniamo i modelli linguistici?
1. Quali sono i tre passaggi per creare un classificatore di testi all'avanguardia?
1. In che modo le 50.000 recensioni di film senza etichetta ci aiutano a creare un migliore classificatore di testo per il set di dati IMDb?
1. Quali sono i tre passaggi per preparare i dati per un modello linguistico?
1. Cos'è la "tokenizzazione"? Perché ne abbiamo bisogno?
1. Nomina tre diversi approcci alla tokenizzazione.
1. Cos'è "xxbos"?
1. Elenca quattro regole che fastai applica al testo durante la tokenizzazione.
1. Perché i caratteri ripetuti vengono sostituiti con un token che mostra il numero di ripetizioni e il carattere ripetuto?
1. Cos'è la "numericizzazione"?
1. Perché potrebbero esserci delle parole sostituite con il token "parola sconosciuta"?
1. Con una dimensione batch pari a 64, la prima riga del tensore che rappresenta il primo batch contiene i primi 64 token per il set di dati. Cosa contiene la seconda riga di quel tensore? Cosa contiene la prima riga del secondo lotto? (Attenzione: gli studenti spesso sbagliano! Assicurati di controllare la tua risposta sul sito web del libro.)
1. Perché abbiamo bisogno del riempimento per la classificazione del testo? Perché non ne abbiamo bisogno per la modellazione del linguaggio?
1. Cosa contiene una matrice di incorporamento per la PNL? Qual è la sua forma?
1. Cos'è la "perplessità"?
1. Perché dobbiamo passare il vocabolario del modello linguistico al blocco dati del classificatore?
1. Cos'è lo "scongelamento graduale"?
1. Perché è probabile che la generazione del testo sia sempre in vantaggio rispetto all’identificazione automatica dei testi generati dalla macchina?

### Ulteriori ricerche

1. Scopri cosa puoi imparare sui modelli linguistici e sulla disinformazione. Quali sono i migliori modelli linguistici oggi? Dai un'occhiata ad alcuni dei loro risultati. Li trovi convincenti? Come potrebbe un cattivo attore utilizzare al meglio un modello del genere per creare conflitto e incertezza?
1. Considerato il limite che i modelli difficilmente sono in grado di riconoscere in modo coerente i testi generati dalle macchine, quali altri approcci potrebbero essere necessari per gestire campagne di disinformazione su larga scala che sfruttano il deep learning?